# AI Subscription Regression Quest
## S03 · Regression I · 90–120 minutes · 100 core XP + 20 bonus

At the end of this month, estimate a fictional user's **total spend in the next 30 days, in EUR**.
One row represents one independently generated user observed once. Inputs describe the current month.
`future_spend` is observed later and must never enter the input features. `user_id` is an identifier.

The data are new synthetic teaching records. You do not need S02's files. The generator deliberately creates
a mostly linear relationship. Results cannot establish real customer behaviour, causal effects or business value.

### How to play
1. Save a private copy in Colab with **File → Save a copy in Drive**.
2. Run setup, then complete M1–M8 in order. Each mission has a guided example and a separate student task.
3. Replace the student placeholders, run the check, then explain your reasoning to a peer or lecturer.
4. After that review, update the corresponding private review flag and rerun the mission check.
5. Run the scoreboard. Core XP and optional bonus XP appear separately.

**Running examples alone earns zero XP.** Starter cells run without errors but leave tasks incomplete.
Checks record each mission once, so retries cannot multiply points. Hints and retries are unlimited.
Points reward valid work and explanations, including results where a baseline wins. Speed and low error earn no bonus.

XP are private, formative feedback. They do not change the official 60% exam / 30% Kaggle / 10% presentation assessment.
The review flags record a learning conversation, not a secure grade or automated proof of understanding.

**LO3:** Build, train, and evaluate supervised learning models (regression and classification) using scikit-learn,
applying rigorous cross-validation. This session develops the regression component.


## Setup
Run this cell once. CPU is enough. No dataset download, API key or paid service is required.
Required packages are NumPy, pandas, Matplotlib and scikit-learn. The cell prints installed versions.
If your local Jupyter lacks a package, install it through your usual environment manager and restart the kernel.


In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
try:
    from IPython.display import display
except ImportError:
    display = print
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, KFold, cross_validate, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__, 'pandas:', pd.__version__)
print('Matplotlib:', matplotlib.__version__, 'scikit-learn:', sklearn.__version__)
SEED = 42
plt.rcParams.update({'figure.figsize': (7, 4), 'axes.grid': True, 'grid.alpha': .2})


## Reproducible fictional data
The generator creates 500 independent users. Some input measurements are missing, while targets remain observed.
It creates future outcomes from complete underlying inputs before injecting missingness.
The artificial formula makes this exercise learnable and is not a claim about AI users.

| Column | Meaning | Role |
| --- | --- | --- |
| user_id | Unique synthetic identifier | Exclude |
| monthly_spend | Current-month spend, EUR | Numeric input |
| weekly_sessions | Mean weekly sessions this month | Numeric input |
| tenure_months | Completed months with the service | Numeric input |
| plan | Current Basic / Pro / Team plan | Categorical input |
| future_spend | Spend over the next 30 days, EUR | Target |


In [ ]:
rng = np.random.default_rng(SEED)
n = 500
monthly = rng.uniform(10, 65, n)
sessions = rng.integers(0, 31, n).astype(float)
tenure = rng.integers(1, 49, n).astype(float)
plan = rng.choice(['Basic', 'Pro', 'Team'], n, p=[.45, .35, .20]).astype(object)
plan_effect = pd.Series(plan).map({'Basic': 0., 'Pro': 8., 'Team': 18.}).to_numpy()
future = 12 + .65 * monthly + 1.8 * sessions + .15 * tenure + plan_effect + rng.normal(0, 6, n)
df = pd.DataFrame({'user_id': np.arange(1000, 1000+n), 'monthly_spend': monthly,
                   'weekly_sessions': sessions, 'tenure_months': tenure,
                   'plan': plan, 'future_spend': future})
for column, count in [('monthly_spend', 40), ('weekly_sessions', 30), ('plan', 15)]:
    df.loc[rng.choice(n, count, replace=False), column] = np.nan
print('Dataset shape:', df.shape)
# Audit structure only. Relationship exploration below uses development data.
display(df.dtypes.rename('dtype').to_frame())
assert df.user_id.is_unique and df.future_spend.notna().all()


## Private XP ledger
Leave review flags False until you have written an explanation and discussed it with a peer or lecturer.
Use the reasoning criteria in each mission. You may revise and resubmit explanations without penalty.
The ledger uses mission IDs as keys, so rerunning a check replaces its record rather than adding another award.
Run setup and this ledger cell before the missions. Rerunning the ledger cell intentionally resets local progress.


In [ ]:
XP_RULES = {'M1': (5,5), 'M2': (5,5), 'M3': (5,5), 'M4': (10,5),
            'M5': (10,5), 'M6': (10,5), 'M7': (5,10), 'M8': (5,5), 'B1': (10,10)}
reviewed = {mission: False for mission in XP_RULES}
xp_ledger = {}

def check_mission(mission, passed, explanation):
    code_max, reasoning_max = XP_RULES[mission]
    code_xp = code_max if passed else 0
    reasoning_xp = reasoning_max if (passed and reviewed[mission] and explanation.strip()) else 0
    xp_ledger[mission] = {'mission':mission, 'code_XP':code_xp, 'reasoning_XP':reasoning_xp,
                           'total_XP':code_xp+reasoning_xp}
    print(f'{mission}: {code_xp+reasoning_xp}/{code_max+reasoning_max} XP')
    if not passed:
        print('Task incomplete or check failed. Read the hint, edit your task and retry.')
    elif reasoning_xp == 0:
        print('Code check passed. Write your explanation and complete the review conversation.')

def scoreboard():
    rows = [xp_ledger.get(m, {'mission':m,'code_XP':0,'reasoning_XP':0,'total_XP':0}) for m in XP_RULES]
    table = pd.DataFrame(rows).set_index('mission')
    core = int(table.loc[[m for m in XP_RULES if m != 'B1'], 'total_XP'].sum())
    bonus = int(table.loc['B1','total_XP'])
    level = ('Regression Ranger' if core == 100 else 'Analyst' if core >= 80 else
             'Evaluator' if core >= 60 else 'Builder' if core >= 30 else 'Explorer')
    display(table)
    print(f'Core: {core}/100 XP | Optional bonus: {bonus}/20 XP | Milestone: {level}')
    return core, bonus


## M1 — Task detective · 10 XP
**Guided idea:** the prediction time determines whether a column is usable. A future invoice can leak the answer.

1. Read the data dictionary.
2. Replace `student_features = None` with a list of the four available input column names.
3. Explain the unit of observation, prediction time, target horizon and why the other two columns are excluded.
4. Run the check. Discuss your explanation, then change `reviewed['M1']` to True in your task cell and rerun.

**Evidence:** your feature list and prediction contract. **Hint:** include measurements available now, never an identifier or future outcome.
**Reasoning review (5 XP):** a peer/lecturer verifies all four contract details and both exclusions.


In [ ]:
# STUDENT TASK M1
student_features = None
answer_M1 = ''
reviewed['M1'] = False


In [ ]:
FEATURES = ['monthly_spend', 'weekly_sessions', 'tenure_months', 'plan']
passed_M1 = isinstance(student_features, list) and len(student_features) == 4 and set(student_features) == set(FEATURES)
check_mission('M1', passed_M1, answer_M1)
# Guided reference keeps later examples runnable while you work on your own task.
X, y = df[FEATURES].copy(), df['future_spend'].copy()


## M2 — Split guardian · 10 XP
1. Run the split cell. It reserves the final test set before learning any transformations.
2. Set `student_sizes` to a tuple `(development_rows, test_rows)` calculated from the split variables.
3. Explain what random_state controls and why shuffled independent-user splits would need revisiting for repeated users or calendar forecasting.
4. Check your answer, then review your reasoning and update the flag.

**Evidence:** dimensions and split rationale. **Hint:** `.shape[0]` or `len(...)` counts rows.
**Reasoning review (5 XP):** identify a seed as a split control, name the untouched test role, and explain one alternative validation design.


In [ ]:
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=.20, random_state=SEED)
assert set(X_dev.index).isdisjoint(X_test.index)
assert len(X_dev) + len(X_test) == len(df)
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
folds = list(cv.split(X_dev, y_dev))
print('Development:', X_dev.shape, 'Test:', X_test.shape)
print('First fold training / validation rows:', len(folds[0][0]), len(folds[0][1]))
display(X_dev.isna().sum().rename('missing_development').to_frame())


In [ ]:
# STUDENT TASK M2
student_sizes = None
answer_M2 = ''
reviewed['M2'] = False


In [ ]:
check_mission('M2', student_sizes == (len(X_dev), len(X_test)), answer_M2)


## M3 — Loss mechanic · 10 XP
The residual is `actual − predicted`. OLS minimises the sum of squared residuals. MSE divides this by n and has the same minimiser.

1. Run the worked example and inspect the residual sign and units.
2. Implement `student_mse(actual, predicted)` using NumPy. It must accept lists or arrays and return a numeric mean squared error.
3. Explain why signed errors cannot simply be averaged and why MSE has squared-euro units.
4. Run the checks on multiple inputs. Review your explanation.

**Hint:** convert each input with `np.asarray(..., dtype=float)`, subtract, square, then take the mean.
**Reasoning review (5 XP):** explain cancellation, larger-error influence, units, and the sign of underprediction.


In [ ]:
worked_actual = np.array([20., 30., 40.])
worked_prediction = np.array([18., 33., 39.])
worked_residual = worked_actual - worked_prediction
display(pd.DataFrame({'actual_EUR':worked_actual, 'prediction_EUR':worked_prediction,
                       'residual_EUR':worked_residual, 'squared_residual_EUR2':worked_residual**2}))
print('SSE:', np.sum(worked_residual**2), 'MSE:', np.mean(worked_residual**2))
print('RMSE:', np.sqrt(np.mean(worked_residual**2)), 'MAE:', np.mean(np.abs(worked_residual)))


In [ ]:
# STUDENT TASK M3
def student_mse(actual, predicted):
    # Replace this placeholder with your calculation.
    return None

answer_M3 = ''
reviewed['M3'] = False


In [ ]:
try:
    test_cases = [([1,3,6], [2,2,4]), ([0,0], [0,0]), ([-2,5], [1,1])]
    passed_M3 = all(np.isclose(student_mse(a,p), mean_squared_error(a,p)) for a,p in test_cases)
except (TypeError, ValueError):
    passed_M3 = False
check_mission('M3', passed_M3, answer_M3)


## M4 — Baseline builder · 15 XP
A training-mean baseline ignores feature values and learns one constant from training targets only.

1. Run the preprocessing factory and baseline CV cells.
2. Fill `student_baseline_mae` with a length-five NumPy array of **positive** fold MAEs obtained from `baseline_cv`.
3. Explain why the mean must be estimated separately inside every fold and why candidates need identical folds.
4. Check your array and review the reasoning.

**Hint:** scikit-learn's loss scorers use a `neg_` prefix so higher is better. Reverse the sign for reporting.
**Reasoning review (5 XP):** name the fold's training targets, the excluded validation/test rows and the fixed comparison protocol.


In [ ]:
NUMERIC = ['monthly_spend', 'weekly_sessions', 'tenure_months']
CATEGORICAL = ['plan']

def make_preprocessor():
    numeric = Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())])
    category = Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                         ('encode', OneHotEncoder(drop='first', handle_unknown='ignore'))])
    return ColumnTransformer([('numeric', numeric, NUMERIC), ('category', category, CATEGORICAL)])

def make_model(estimator):
    return Pipeline([('prepare', make_preprocessor()), ('regressor', estimator)])

SCORING = {'mae':'neg_mean_absolute_error', 'mse':'neg_mean_squared_error', 'r2':'r2'}
baseline_pipeline = make_model(DummyRegressor(strategy='mean'))
baseline_cv = cross_validate(baseline_pipeline, X_dev, y_dev, cv=folds,
                             scoring=SCORING, error_score='raise')
print('Raw neg_MAE scores:', baseline_cv['test_mae'])


In [ ]:
# STUDENT TASK M4
student_baseline_mae = None
answer_M4 = ''
reviewed['M4'] = False


In [ ]:
try:
    passed_M4 = np.asarray(student_baseline_mae).shape == (5,) and np.allclose(student_baseline_mae, -baseline_cv['test_mae'])
except (TypeError, ValueError):
    passed_M4 = False
check_mission('M4', passed_M4, answer_M4)


## M5 — Pipeline engineer · 15 XP
The guided candidate appends LinearRegression to the S02 preprocessing structure.
Numeric medians, scaling statistics and categories must be learned inside each fold.

1. Run the guided candidate. Inspect the first fold's medians against its training rows.
2. Create `student_pipeline` yourself with steps named `prepare` and `regressor`.
   Use `make_preprocessor()` and a new `LinearRegression()` instance.
3. Explain why fitting the preprocessor once on all development rows before CV would leak information.
4. Run the check, which clones your pipeline and fits only the first fold's training rows. Review your explanation.

**Hint:** `Pipeline` accepts a list of `(step_name, object)` pairs. The factory builds an unfitted preprocessor.
**Reasoning review (5 XP):** locate where fit occurs, identify learned quantities, and explain how validation rows stay excluded.


In [ ]:
linear_pipeline = make_model(LinearRegression())
linear_cv = cross_validate(linear_pipeline, X_dev, y_dev, cv=folds, scoring=SCORING,
                           return_estimator=True, error_score='raise')
train_positions, validation_positions = folds[0]
fold_imputer = linear_cv['estimator'][0].named_steps['prepare'].named_transformers_['numeric'].named_steps['impute']
expected_medians = X_dev.iloc[train_positions][NUMERIC].median().to_numpy()
display(pd.DataFrame({'feature':NUMERIC, 'fold_imputer_median':fold_imputer.statistics_,
                       'fold_training_median':expected_medians}))
assert np.allclose(fold_imputer.statistics_, expected_medians)


In [ ]:
# STUDENT TASK M5
student_pipeline = None
answer_M5 = ''
reviewed['M5'] = False


In [ ]:
passed_M5 = False
if isinstance(student_pipeline, Pipeline):
    try:
        student_fold = clone(student_pipeline).fit(X_dev.iloc[train_positions], y_dev.iloc[train_positions])
        student_prepare = student_fold.named_steps['prepare']
        num_steps = student_prepare.named_transformers_['numeric'].named_steps
        student_predictions = student_fold.predict(X_dev.iloc[validation_positions])
        passed_M5 = (isinstance(student_fold.named_steps['regressor'], LinearRegression)
                     and isinstance(student_prepare, ColumnTransformer)
                     and isinstance(num_steps['scale'], StandardScaler)
                     and np.allclose(num_steps['impute'].statistics_, expected_medians)
                     and np.allclose(student_predictions, linear_cv['estimator'][0].predict(X_dev.iloc[validation_positions])))
    except (TypeError, ValueError, KeyError, AttributeError):
        passed_M5 = False
check_mission('M5', passed_M5, answer_M5)


## M6 — Validation analyst · 15 XP
**Predeclared rule:** select the lower mean development CV MAE. Exact ties favour the mean baseline.
MAE is the selection metric even though OLS trains by minimising squared loss.

1. Run the summary cell. Both candidates use the same five folds and fold-local preprocessing.
2. Create `student_fold_table` with five rows and columns `baseline_MAE` and `linear_MAE` from the returned CV scores.
3. Explain the chosen candidate using the actual means and fold spread. State why the spread is not a confidence interval.
4. Run the check and discuss your interpretation. Do not inspect the final test to choose a candidate.

**Hint:** use `pd.DataFrame` and reverse the scorer signs. For each fold's RMSE, square-root that fold's positive MSE first.
**Reasoning review (5 XP):** cite the observed results, explain the selection rule and its uncertainty, and distinguish validation from final test.


In [ ]:
def cv_summary(result):
    mae = -result['test_mae']
    rmse = np.sqrt(-result['test_mse'])
    return {'CV_MAE_EUR':mae.mean(), 'fold_MAE_sd_EUR':mae.std(ddof=1),
            'mean_fold_RMSE_EUR':rmse.mean(), 'mean_fold_R2':result['test_r2'].mean()}

comparison = pd.DataFrame({'Mean baseline':cv_summary(baseline_cv),
                           'Linear regression':cv_summary(linear_cv)}).T
display(comparison.round(3))
selected_name = ('Linear regression' if comparison.loc['Linear regression','CV_MAE_EUR'] <
                  comparison.loc['Mean baseline','CV_MAE_EUR'] else 'Mean baseline')
print('Locked selection from development CV:', selected_name)

# Development diagnostics only. Five additional fits generate out-of-fold predictions.
oof_prediction = cross_val_predict(linear_pipeline, X_dev, y_dev, cv=folds)
oof_residual = y_dev.to_numpy() - oof_prediction
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(y_dev, oof_prediction, alpha=.45)
limits = [min(y_dev.min(), oof_prediction.min()), max(y_dev.max(), oof_prediction.max())]
axes[0].plot(limits, limits, color='black', linestyle='--', label='Equality reference')
axes[0].set(xlabel='Observed next-30-day spend (EUR)', ylabel='OOF predicted spend (EUR)', title='Development predictions')
axes[0].legend()
axes[1].scatter(oof_prediction, oof_residual, alpha=.45)
axes[1].axhline(0, color='black', linestyle='--')
axes[1].set(xlabel='OOF predicted spend (EUR)', ylabel='Actual minus predicted (EUR)', title='Development residuals')
plt.tight_layout()
plt.show()
print('Text alternative: left compares held-out development predictions with actual values.')
print('Right shows signed errors around zero. Look for curvature, unequal spread and isolated errors.')


In [ ]:
# STUDENT TASK M6
student_fold_table = None
answer_M6 = ''
reviewed['M6'] = False


In [ ]:
passed_M6 = False
if isinstance(student_fold_table, pd.DataFrame):
    try:
        passed_M6 = (len(student_fold_table) == 5
                     and np.allclose(student_fold_table['baseline_MAE'], -baseline_cv['test_mae'])
                     and np.allclose(student_fold_table['linear_MAE'], -linear_cv['test_mae']))
    except (KeyError, TypeError, ValueError):
        passed_M6 = False
check_mission('M6', passed_M6, answer_M6)


## M7 — Evidence reporter · 15 XP
The candidate is now locked. This is the **single final test evaluation stage**, with a fixed baseline reference.
Test results do not select the candidate. If you change a design because of these results, you need a new evaluation plan.

1. Run the final refit/evaluation cell. Both locked estimators fit only on development data.
2. Fill `student_test_metrics` with a dict containing `MAE_EUR`, `RMSE_EUR` and `R2` for `selected_test_prediction`.
3. Write a short model card: task and units, selection protocol, actual test figures, comparison with the baseline,
   one error-cost consideration and two limitations. Say when the baseline wins if it does.
4. Run the check. A reviewer verifies the report's evidence and limitations before you update the flag.

**Hint:** use the imported metric functions and `np.sqrt(mean_squared_error(...))`.
**Reasoning review (10 XP):** the model card must cover all six requested items, with no claim of real-market validity,
causation, statistical significance from five folds alone, or guaranteed individual error bounds.


In [ ]:
def metric_report(actual, predicted):
    return {'MAE_EUR':mean_absolute_error(actual,predicted),
            'RMSE_EUR':np.sqrt(mean_squared_error(actual,predicted)), 'R2':r2_score(actual,predicted)}

locked_candidate = linear_pipeline if selected_name == 'Linear regression' else baseline_pipeline
selected_model = clone(locked_candidate).fit(X_dev, y_dev)
baseline_final = clone(baseline_pipeline).fit(X_dev, y_dev)
selected_test_prediction = selected_model.predict(X_test)
baseline_test_prediction = baseline_final.predict(X_test)
test_table = pd.DataFrame({'Selected: '+selected_name:metric_report(y_test, selected_test_prediction),
                           'Fixed mean baseline':metric_report(y_test, baseline_test_prediction)}).T
display(test_table.round(3))
print('Final test stage complete. Preserve these results; do not tune against them.')


In [ ]:
# STUDENT TASK M7
student_test_metrics = None
answer_M7 = ''
reviewed['M7'] = False


In [ ]:
expected_metrics = metric_report(y_test, selected_test_prediction)
try:
    passed_M7 = (isinstance(student_test_metrics, dict) and
                  all(np.isclose(student_test_metrics[k], v) for k,v in expected_metrics.items()))
except (KeyError, TypeError, ValueError):
    passed_M7 = False
check_mission('M7', passed_M7, answer_M7)


## M8 — Prediction guide · 10 XP
1. Run the example prediction for a new fictional user. The fitted pipeline handles preprocessing.
2. Create your own **one-row** `student_new_user` DataFrame with the four input columns.
   Choose values within the generator's range: spend 10–65 EUR, sessions 0–30, tenure 1–48, and a known plan.
3. Set `student_new_prediction` to the numeric result of `selected_model.predict(...)` for your row. Do not refit.
4. Explain the horizon, units, feature availability and why this point estimate has no calibrated uncertainty interval.
5. Run the check, review your explanation and view the scoreboard.

**Hint:** pandas needs a list containing one dictionary to create a one-row table.
**Reasoning review (5 XP):** cover the horizon, units, no-refit rule, synthetic scope and absence of an individual error guarantee.


In [ ]:
new_user = pd.DataFrame([{'monthly_spend':35., 'weekly_sessions':12., 'tenure_months':10., 'plan':'Pro'}])
new_prediction = float(selected_model.predict(new_user)[0])
display(new_user)
print(f'Fictional next-30-day spend estimate: EUR {new_prediction:.2f}')
print('This point estimate has no calibrated prediction interval and is not a real customer forecast.')


In [ ]:
# STUDENT TASK M8
student_new_user = None
student_new_prediction = None
answer_M8 = ''
reviewed['M8'] = False


In [ ]:
passed_M8 = False
if isinstance(student_new_user, pd.DataFrame) and len(student_new_user) == 1:
    try:
        row = student_new_user.iloc[0]
        valid_input = (set(student_new_user.columns) == set(FEATURES)
                        and 10 <= row.monthly_spend <= 65 and 0 <= row.weekly_sessions <= 30
                        and 1 <= row.tenure_months <= 48 and row.plan in ['Basic','Pro','Team'])
        passed_M8 = valid_input and np.isclose(student_new_prediction, selected_model.predict(student_new_user)[0])
    except (KeyError, AttributeError, TypeError, ValueError):
        passed_M8 = False
check_mission('M8', passed_M8, answer_M8)
core_xp, bonus_xp = scoreboard()


## B1 — Optional median baseline · 20 bonus XP
Keep the locked core comparison and final test result unchanged.

1. Build a new pipeline with `DummyRegressor(strategy='median')`.
2. Use only `X_dev`, `y_dev` and the existing `folds` to obtain five validation MAEs.
3. Store the positive errors in `student_median_mae`.
4. Explain how a median baseline relates to absolute error and compare its actual CV result with the mean baseline.
5. Run the check and discuss your explanation. This is additional development exploration, not a new test-based selection.

**Hint:** follow the baseline CV example and change only its strategy. Use the same scorer and folds.
**Reasoning review (10 XP):** connect median to absolute loss, cite the observed comparison and explain why the test stage remains locked.


In [ ]:
# STUDENT TASK B1
student_median_mae = None
answer_B1 = ''
reviewed['B1'] = False


In [ ]:
median_expected = np.array([mean_absolute_error(y_dev.iloc[va],
                              np.repeat(y_dev.iloc[tr].median(), len(va))) for tr,va in folds])
try:
    passed_B1 = np.asarray(student_median_mae).shape == (5,) and np.allclose(student_median_mae, median_expected)
except (TypeError, ValueError):
    passed_B1 = False
check_mission('B1', passed_B1, answer_B1)
core_xp, bonus_xp = scoreboard()


## Finish and reflect
Save your private notebook, including outputs, explanations and the scoreboard. Share through the lecturer's normal
institutional channel only if requested. Do not publish student names, submissions, review records or XP.

**Exit ticket:** What does a positive residual mean? Which rows may a fold's imputer use? Why can a baseline be the best choice?

**Fit count reminder:** `train_test_split` performs zero fits. Five-fold CV performs five fits per candidate.
The OOF diagnostic call fits the linear candidate five more times. The two final `.fit()` calls refit locked models
on development data. Student checks and the optional bonus can add fits. None of these counts is an epoch count.

**Next session:** S04 explores polynomial models and regularisation. Keep this tutorial's final test result as recorded.

### Primary reading
- [LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)
- [DummyRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyRegressor.html)
- [Cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html)
- [Metrics and scoring](https://scikit-learn.org/stable/modules/model_evaluation.html)
- [Leakage and pipelines](https://scikit-learn.org/stable/common_pitfalls.html)

Original synthetic teaching material. Local execution checked 4 November 2026. Colab/browser behaviour and professor review remain pending.
